In [1]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.manifold import MDS
import pandas as pd
import numpy as np
import nltk
import matplotlib.pyplot as plt
import itertools
import re





In [2]:

# NLTK resources (run once)

nltk.download("punkt")
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")
nltk.download("averaged_perceptron_tagger")
nltk.download("universal_tagset")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.
[nltk_data] Downloading package universal_tagset to /root/nltk_data...
[nltk_data]   Unzipping taggers/universal_tagset.zip.


True

In [3]:
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [4]:
nltk.download("averaged_perceptron_tagger_eng")

[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.


True

In [5]:

# Load + filter English

df = pd.read_excel("reviews_annotated_lda_labels.xlsx")

#
TEXT_COL = "Review"
LABEL_COL = "Service"

df = df[df["Language"] == "English"].copy()
df = df[[TEXT_COL, LABEL_COL]].dropna()

# Label to 0/1 if it's Yes/No, used the manual labeling
df[LABEL_COL] = (
    df[LABEL_COL]
    .replace({"Yes": 1, "No": 0, "yes": 1, "no": 0})
    .infer_objects(copy=False)
    .astype(int)
)


# cleaning step BEFORE tokenization
# remove URLs, punctuation/symbols, numbers

def basic_clean(s: str) -> str:
    s = str(s).lower()
    s = re.sub(r"http\S+|www\.\S+", " ", s)       # remove URLs
    s = re.sub(r"[^a-z\s]", " ", s)              # keep letters + whitespace only
    s = re.sub(r"\s+", " ", s).strip()
    return s

df["Review_lower"] = df[TEXT_COL].apply(basic_clean)


# Tokenize + POS tag

stop_words = nltk.corpus.stopwords
df["tokenized"] = df["Review_lower"].apply(nltk.word_tokenize)

# universal tagset like your example
df["position"] = df["tokenized"].apply(lambda toks: nltk.pos_tag(toks, tagset="universal"))


# Map universal POS -> WordNet POS for lemmatizer

from nltk.corpus import wordnet
from nltk.stem import WordNetLemmatizer
wnl = WordNetLemmatizer()

def to_wordnet_pos(universal_tag: str):
    if universal_tag == "NOUN":
        return wordnet.NOUN
    if universal_tag == "VERB":
        return wordnet.VERB
    if universal_tag == "ADJ":
        return wordnet.ADJ
    if universal_tag == "ADV":
        return wordnet.ADV
    return None  # for PRON, DET, ADP, etc.


# Lemmatize (POS-aware) + stopword removal + trimming

def lemmatize_with_pos(pos_pairs):
    out = []
    for word, utag in pos_pairs:
        wn_pos = to_wordnet_pos(utag)
        if wn_pos is None:
            lemma = wnl.lemmatize(word)          # fallback
        else:
            lemma = wnl.lemmatize(word, wn_pos)
        out.append(lemma)
    return out

df["tokenized_lemma"] = df["position"].apply(lemmatize_with_pos)

df["tokenized_lemma_no_stop"] = df["tokenized_lemma"].apply(
    lambda toks: [w for w in toks if w not in stop_words.words("english") and len(w) > 2]
)

# Join back to cleaned text for vectorizers
df["text_clean"] = df["tokenized_lemma_no_stop"].apply(lambda toks: " ".join(toks))

# Drop empty after cleaning
df = df[df["text_clean"].str.len() > 0].copy()


# Build TF-IDF matrices

tfidf_uni = TfidfVectorizer(ngram_range=(1, 1), max_features=5000)
X_uni = tfidf_uni.fit_transform(df["text_clean"])

tfidf_bi = TfidfVectorizer(ngram_range=(1, 2), max_features=5000)
X_bi = tfidf_bi.fit_transform(df["text_clean"])

y = df[LABEL_COL].values

print("Rows:", len(df))
print("Unigram TF-IDF shape:", X_uni.shape)
print("Bigram  TF-IDF shape:", X_bi.shape)
print("y shape:", y.shape)


FileNotFoundError: [Errno 2] No such file or directory: 'reviews_annotated_lda_labels.xlsx'

In [ ]:
### Because TF-IDF representations are high-dimensional
### and sparse, cosine distance is more appropriate as it
### measures similarity based on vector direction rather than
### magnitude, reducing sensitivity to document length
### differences.

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score



# train/test split
Xuni_train, Xuni_test, y_train, y_test = train_test_split(
    X_uni, y, test_size=0.2, random_state=42, stratify=y
)

Xbi_train, Xbi_test, _, _ = train_test_split(
    X_bi, y, test_size=0.2, random_state=42, stratify=y
)

feature_sets = {
    "Unigram_TFIDF": (Xuni_train, Xuni_test),
    "Uni+Bigram_TFIDF": (Xbi_train, Xbi_test)
}

k_values = [1,3,5]

results = []

for fname, (Xtrain, Xtest) in feature_sets.items():

    for k in k_values:

        model = KNeighborsClassifier(
            n_neighbors=k,
            metric="cosine"
        )

        # train model
        model.fit(Xtrain, y_train)
        y_pred = model.predict(Xtest)

        # evaluation metrics
        precision = precision_score(y_test, y_pred, pos_label=1)
        recall = recall_score(y_test, y_pred, pos_label=1)
        f1 = f1_score(y_test, y_pred, pos_label=1)

        # cross-validation on training set
        cv_scores = cross_val_score(model, Xtrain, y_train, cv=5, scoring="f1")

        results.append({
            "features": fname,
            "k": k,
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "cv_f1_mean": cv_scores.mean()
        })

results_df = pd.DataFrame(results).sort_values(by="f1", ascending=False)

print(results_df)

In [ ]:
### Unigram TF-IDF achieved the highest accuracy,
### suggesting that single-word features were sufficient
### to capture service-related patterns in the dataset.
### Although bigrams introduce contextual information,
### they significantly increase feature sparsity, which may
### negatively impact KNN performance in smaller datasets.
### The best performance occurred at k=1, indicating that
### nearest-neighbor similarity based on TF-IDF
###representations effectively captures semantic similarity
### between reviews.